In [ ]:
### Imports + Config
import json
import re
import os
from pathlib import Path
from itertools import permutations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv

## Für Debugs
from collections import defaultdict
from difflib import SequenceMatcher

# Pandas-Options:
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 160)

load_dotenv()

repo_root_env = os.getenv("REPO_ROOT")
assert repo_root_env, "REPO_ROOT ist nicht gesetzt in .env"
REPO_ROOT = Path(repo_root_env).expanduser().resolve()
assert REPO_ROOT.exists(), f"REPO_ROOT existiert nicht: {REPO_ROOT}"

RESULTS_DIR = REPO_ROOT / "results"
assert RESULTS_DIR.exists(), f"RESULTS_DIR existiert nicht: {RESULTS_DIR}"

FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

### Ausgegebenes und annotiertes Label-Muster als Regex
SPEAKER_PATTERN = re.compile(r'\[Sprecher (\d+)\]:\s*')

# Pipeline-Übersicht, macht Auswertung einfacher
PIPELINE_ORDER = ["A1_raw", "A1_segmented", "A2_raw", "A2_segmented", "B", "C", "D"]
PIPELINE_LABELS = {
    "A1_raw": "A₁: MedGemma (raw)",
    "A1_seg": "A1: MedGemma (seg)",
    "A2_raw": "A₂: Llama-3.1 (raw)",
    "A2_seg": "A2: Llama-3.1 (seg)",
    "B": "B: Pya-Seg",
    "C": "C: Pya-FW",
    "D": "D: Pya-WX",
}
CONDITION_ORDER = ["unknown", "known"]
CONDITION_COLORS = {"unknown": "#D32F2F", "known": "#1565C0"}

In [ ]:
### Parser
# Parst '[Sprecher N]: text' Format.
# Returns: (tokens, labels) — gleich lang, ein Label pro Wort.
def parse_dialogue(text: str) -> tuple[list[str], list[str]]:

    tokens, labels = [], []
    current_speaker = None

    for line in text.strip().split('\n'):
        line = line.strip()
        if not line:
            continue

        m = SPEAKER_PATTERN.match(line)
        if m:
            current_speaker = f"S{m.group(1)}"
            content = SPEAKER_PATTERN.sub('', line).strip()
        else:
            # Zeile ohne Speaker-Label → vorherigen Speaker fortsetzen
            if current_speaker is None:
                current_speaker = "S0"
            content = line
        
        words = content.split()
        tokens.extend(words)
        labels.extend([current_speaker] * len(words))
    
    return tokens, labels

## Speaker-Labels entfernen
def strip_speaker_labels(text: str) -> list[str]:
    return SPEAKER_PATTERN.sub("", text).strip().split()

In [ ]:
### Prüfung, ob LLM Tokenstream verändert hat

def validate_token_stream(ref_text: str, hyp_text: str) -> dict:

    # Speaker-Labels entfernen
    ref_tokens = strip_speaker_labels(ref_text)
    hyp_tokens = strip_speaker_labels(hyp_text)
    
    if ref_tokens == hyp_tokens:
        return {
            "valid": True,
            "n_tokens": len(ref_tokens),
            "n_ref": len(ref_tokens),
            "n_hyp": len(hyp_tokens),
            "insertions": 0,
            "deletions": 0,
            "substitutions": 0,
            "token_error_rate": 0.0,
        }

    sm = SequenceMatcher(None, ref_tokens, hyp_tokens)
    ins, dels, subs = 0, 0, 0

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "insert":
            ins += j2 - j1
        elif tag == "delete":
            dels += i2 - i1
        elif tag == "replace":
            subs += max(i2 - i1, j2 - j1)

    return {
        "valid": False,
        "n_ref": len(ref_tokens),
        "n_hyp": len(hyp_tokens),
        "insertions": ins,
        "deletions": dels,
        "substitutions": subs,
        "token_error_rate": round((ins + dels + subs) / max(len(ref_tokens), 1), 4),
    }


def first_token_mismatch(ref_text: str, hyp_text: str, context: int = 8) -> dict:
    ref_tokens = strip_speaker_labels(ref_text)
    hyp_tokens = strip_speaker_labels(hyp_text)

    sm = SequenceMatcher(None, ref_tokens, hyp_tokens)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        return {
            "tag": tag,
            "ref_span": (i1, i2),
            "hyp_span": (j1, j2),
            "ref_tokens": ref_tokens[i1:i2],
            "hyp_tokens": hyp_tokens[j1:j2],
            "ref_context": ref_tokens[max(0, i1 - context):min(len(ref_tokens), i2 + context)],
            "hyp_context": hyp_tokens[max(0, j1 - context):min(len(hyp_tokens), j2 + context)],
        }

    return {"tag": "equal"}

In [ ]:
### Optimale Speaker-Zuordnung (Permutationen beachten)
def best_speaker_mapping(pred_labels: list[str], ref_labels: list[str]) -> dict:
    """
    Findet optimale 1:1-Zuordnung von pred→ref Speakern via Brute-Force.
    Funktioniert für <=4 Sprecher (4! = 24 Permutationen).
    Bei mehr pred-Speakern als Referenz-Speakern bleiben zusätzliche Speaker unmapped und zählen als Fehler.
    """
    pred_speakers = sorted(set(pred_labels))
    ref_speakers = sorted(set(ref_labels))

    best_mapping = None
    best_correct = -1

    if len(pred_speakers) <= len(ref_speakers):
        for perm in permutations(ref_speakers):
            mapping = {p: r for p, r in zip(pred_speakers, perm[:len(pred_speakers)])}
            correct = sum(
                1 for p, r in zip(pred_labels, ref_labels)
                if mapping.get(p) == r
            )
            if correct > best_correct:
                best_correct = correct
                best_mapping = mapping
    else:
        for perm in permutations(pred_speakers):
            mapping = {p: r for p, r in zip(perm[:len(ref_speakers)], ref_speakers)}
            for p in pred_speakers:
                if p not in mapping:
                    mapping[p] = f"UNMAPPED_{p}"
            correct = sum(
                1 for p, r in zip(pred_labels, ref_labels)
                if mapping.get(p) == r
            )
            if correct > best_correct:
                best_correct = correct
                best_mapping = mapping

    if best_mapping is None:
        best_mapping = {p: p for p in pred_speakers}

    return best_mapping

In [ ]:
### WSER berechnen
def compute_wser(pred_text: str, ref_text: str) -> dict:

    pred_tokens, pred_labels = parse_dialogue(pred_text)
    ref_tokens, ref_labels = parse_dialogue(ref_text)
    
    # Token-Stream muss identisch sein
    if pred_tokens != ref_tokens:
        return {
            "wser": None,
            "error": f"Token mismatch: {len(pred_tokens)} pred vs {len(ref_tokens)} ref",
            "n_total": len(ref_tokens),
            "n_pred_tokens": len(pred_tokens),
            "n_speakers_pred": len(set(pred_labels)),
            "n_speakers_ref": len(set(ref_labels)),
            "mapping": None,
        }
    
    mapping = best_speaker_mapping(pred_labels, ref_labels)
    mapped_pred = [mapping.get(l, l) for l in pred_labels]
    
    n_total = len(ref_labels)
    n_correct = sum(1 for p, r in zip(mapped_pred, ref_labels) if p == r)
    n_errors = n_total - n_correct
    
    return {
        "wser": round(n_errors / n_total, 4) if n_total > 0 else 0.0,
        "n_total": n_total,
        "n_correct": n_correct,
        "n_errors": n_errors,
        "mapping": mapping,
        "n_speakers_pred": len(set(pred_labels)),
        "n_speakers_ref": len(set(ref_labels)),
    }

In [ ]:
### Boundary-WSER berechnen (im Fenster +- k Wörter um Sprecherwechsel)
def compute_boundary_wser(pred_text: str, ref_text: str, k: int = 2) -> dict:

    pred_tokens, pred_labels = parse_dialogue(pred_text)
    ref_tokens, ref_labels = parse_dialogue(ref_text)

    if pred_tokens != ref_tokens:
        return {
            "boundary_wser": None,
            "n_boundary_words": None,
            "n_boundary_correct": None,
            "n_turns": None,
            "error": "Token mismatch",
        }
    
    mapping = best_speaker_mapping(pred_labels, ref_labels)
    mapped_pred = [mapping.get(l, l) for l in pred_labels]
    
    # Positionen der Sprecherwechsel in der Referenz finden
    boundary_positions = set()
    for i in range(1, len(ref_labels)):
        if ref_labels[i] != ref_labels[i-1]:
            # Fenster +-k um den Wechsel
            for j in range(max(0, i - k), min(len(ref_labels), i + k + 1)):
                boundary_positions.add(j)
    
    if not boundary_positions:
        return {
            "boundary_wser": 0.0,
            "n_boundary_words": 0,
            "n_boundary_correct": 0,
            "n_turns": 0,
            "error": None,
        }    
    n_boundary = len(boundary_positions)
    n_correct = sum(1 for i in boundary_positions if mapped_pred[i] == ref_labels[i])
    
    return {
        "boundary_wser": round((n_boundary - n_correct) / n_boundary, 4),
        "n_boundary_words": n_boundary,
        "n_boundary_correct": n_correct,
        "n_turns": sum(1 for i in range(1, len(ref_labels)) if ref_labels[i] != ref_labels[i-1]),
        "error": None,
    }

In [ ]:
### RTF-Berechnung aus meta.json
def compute_rtf(meta: dict, run_key: str) -> dict:
    audio_dur = meta.get("audio_duration_s")
    if not audio_dur or audio_dur == 0:
        return {"rtf_total": None, "rtf_post_asr": None, "components": {}}

    run = meta.get("runs", {}).get(run_key, {})
    sec_e2e = float(run.get("seconds_e2e", 0.0) or 0.0)
    sec_post = float(run.get("seconds_post_asr", 0.0) or 0.0)

    sec_norm = float(meta.get("seconds", {}).get("normalize", 0.0) or 0.0)
    sec_asr = float(meta.get("seconds", {}).get("asr", 0.0) or 0.0)
    sec_align = float(meta.get("seconds", {}).get("whisperx_align", 0.0) or 0.0)

    is_aligned = ("word_aligned" in run_key.lower()) or ("wx" in run_key.lower())
    sec_diarize = max(0.0, sec_post - (sec_align if is_aligned else 0.0))

    return {
        "rtf_total": round(sec_e2e / audio_dur, 4),
        "rtf_post_asr": round(sec_post / audio_dur, 4),
        "components": {
            "normalization": round(sec_norm / audio_dur, 4),
            "asr": round(sec_asr / audio_dur, 4),
            "alignment": round(sec_align / audio_dur, 4) if is_aligned else 0.0,
            "diarization": round(sec_diarize / audio_dur, 4),
        },
    }

In [ ]:
### Batch-Aggregation
# Standard: echte manuelle Referenz verwenden.
# Nur zum Debuggen von Pipeline-Struktur auf True setzen, wenn noch keine manuelle Referenz vorhanden.
USE_PSEUDO_REF = True


def parse_pipeline(run_key: str) -> str:
    rk = run_key.lower()

    # LLM-Pipelines
    if "medgemma" in rk:
        base = "A1"
        if "segmented" in rk:
            return f"{base}_segmented"
        if "raw" in rk:
            return f"{base}_raw"

    if "llama" in rk or "meta-llama" in rk:
        base = "A2"
        if "segmented" in rk:
            return f"{base}_segmented"
        if "raw" in rk:
            return f"{base}_raw"
        
    # Pyannote-Pipelines
    if "word_aligned" in rk or "word_alligned" in rk or "pya-wx" in rk:
        return "D"
    if "diarized_word" in rk or "pya-fw" in rk:
        return "C"
    if "diarized_segment" in rk or "pya-seg" in rk:
        return "B"
    return run_key


def parse_condition(run_key: str) -> str:
    rk = run_key.lower()
    if "unknown" in rk:
        return "unknown"
    if "known" in rk:
        return "known"
    return "other"


def add_error(existing: str | None, new_msg: str) -> str:
    if existing in (None, "", "None"):
        return new_msg
    return f"{existing} | {new_msg}"


rows = []
skipped = []

for result_dir in sorted(RESULTS_DIR.iterdir()):
    if not result_dir.is_dir() or result_dir.name == "figures":
        continue

    meta_path = result_dir / "meta.json"
    transcript_path = result_dir / "transcript.txt"
    ref_path = (
        result_dir / "Pyannote_diarized_segment_known.txt"
        if USE_PSEUDO_REF
        else result_dir / "diarization_ref.txt"
    )

    missing = []
    if not meta_path.exists():
        missing.append("meta.json")
    if not transcript_path.exists():
        missing.append("transcript.txt")
    if not ref_path.exists():
        missing.append(ref_path.name)

    if missing:
        skipped.append({"file": result_dir.name, "missing": missing})
        continue

    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    transcript = transcript_path.read_text(encoding="utf-8")
    ref_text = ref_path.read_text(encoding="utf-8")

    try:
        ref_tokens, ref_labels = parse_dialogue(ref_text)
    except Exception as e:
        skipped.append({"file": result_dir.name, "missing": [f"ref parse error: {e!r}"]})
        continue

    spk_match = re.search(r"_(\d)S", result_dir.name)
    n_speakers_from_filename = int(spk_match.group(1)) if spk_match else None

    for run_key, run_info in meta.get("runs", {}).items():
        pipeline = parse_pipeline(run_key)
        condition = parse_condition(run_key)
        output_path = result_dir / f"{run_key}.txt"

        row = {
            "file": result_dir.name,
            "run_key": run_key,
            "pipeline": pipeline,
            "condition": condition,
            "output_exists": output_path.exists(),
            "nonempty": None,
            "valid": False,
            "valid_vs_transcript": None,
            "n_ref_eval": None,
            "n_hyp_eval": None,
            "n_ref_transcript": None,
            "n_hyp_transcript": None,
            "insertions": None,
            "deletions": None,
            "substitutions": None,
            "token_error_rate": None,
            "wser": None,
            "boundary_wser": None,
            "n_words": None,
            "n_boundary_words": None,
            "n_turns": None,
            "n_speakers_from_filename": n_speakers_from_filename,
            "known_num_speakers_meta": meta.get("known_num_speakers"),
            "n_speakers_ref_eval": len(set(ref_labels)),
            "n_speakers_pred": None,
            "audio_duration_s": meta.get("audio_duration_s"),
            "rtf_total": None,
            "rtf_post_asr": None,
            "rtf_normalization": None,
            "rtf_asr": None,
            "rtf_alignment": None,
            "rtf_diarization": None,
            "ok": bool(run_info.get("ok", False)),
            "error": run_info.get("error"),
            "ref_source": ref_path.name,
            "mapping": None,
        }

        try:
            rtf = compute_rtf(meta, run_key)
            row["rtf_total"] = rtf["rtf_total"]
            row["rtf_post_asr"] = rtf["rtf_post_asr"]
            for comp_key, comp_val in rtf.get("components", {}).items():
                row[f"rtf_{comp_key}"] = comp_val
        except Exception as e:
            row["error"] = add_error(row["error"], f"RTF: {e!r}")

        if not output_path.exists():
            row["error"] = add_error(row["error"], "missing output file")
            rows.append(row)
            continue

        hyp_text = output_path.read_text(encoding="utf-8")
        row["nonempty"] = bool(hyp_text.strip())
        if not row["nonempty"]:
            row["error"] = add_error(row["error"], "empty output")
            rows.append(row)
            continue

        # Primärer Validity-Check: gegen Evaluationsreferenz.
        validation = validate_token_stream(ref_text, hyp_text)
        row["valid"] = validation["valid"]
        row["n_ref_eval"] = validation.get("n_ref", validation.get("n_tokens"))
        row["n_hyp_eval"] = validation.get("n_hyp", validation.get("n_tokens"))
        row["insertions"] = validation.get("insertions", 0)
        row["deletions"] = validation.get("deletions", 0)
        row["substitutions"] = validation.get("substitutions", 0)
        row["token_error_rate"] = validation.get("token_error_rate", 0.0)

        # Sekundärer Diagnose-Check: gegen kanonisches Transcript.
        validation_transcript = validate_token_stream(transcript, hyp_text)
        row["valid_vs_transcript"] = validation_transcript["valid"]
        row["n_ref_transcript"] = validation_transcript.get("n_ref", validation_transcript.get("n_tokens"))
        row["n_hyp_transcript"] = validation_transcript.get("n_hyp", validation_transcript.get("n_tokens"))

        if row["valid"]:
            wser_result = compute_wser(hyp_text, ref_text)
            if wser_result.get("error") is None:
                row["wser"] = wser_result["wser"]
                row["n_words"] = wser_result["n_total"]
                row["n_speakers_pred"] = wser_result["n_speakers_pred"]
                row["n_speakers_ref_eval"] = wser_result["n_speakers_ref"]
                row["mapping"] = str(wser_result["mapping"])

                bwser_result = compute_boundary_wser(hyp_text, ref_text, k=2)
                if bwser_result.get("error") is None:
                    row["boundary_wser"] = bwser_result["boundary_wser"]
                    row["n_boundary_words"] = bwser_result["n_boundary_words"]
                    row["n_turns"] = bwser_result["n_turns"]
                else:
                    row["error"] = add_error(row["error"], f"Boundary-WSER: {bwser_result['error']}")
            else:
                row["error"] = add_error(row["error"], f"WSER: {wser_result['error']}")

        rows.append(row)

df = pd.DataFrame(rows)

print("=" * 70)
print("Evaluation Summary")
print("=" * 70)
print(f"Referenz-Modus:   {'PSEUDO (Pyannote-Segment)' if USE_PSEUDO_REF else 'MANUELL (diarization_ref.txt)'}")
print(f"Result-Ordner:    {df['file'].nunique() if len(df) else 0}")
print(f"Runs gesamt:      {len(df)}")
print(f"Valide Runs:      {int(df['valid'].fillna(False).sum())} / {len(df)}")
print(f"Runs mit WSER:    {int(df['wser'].notna().sum())}")
print("=" * 70)

if skipped:
    print("\nÜbersprungene Ordner:")
    for item in skipped:
        print(f"  {item['file']}: {item['missing']}")

if len(df):
    print("\nValidity Rate pro Pipeline x Condition:")
    validity = (
        df.groupby(["pipeline", "condition"], dropna=False)["valid"]
        .agg(["sum", "count", "mean"])
        .rename(columns={"sum": "valid_runs", "count": "total_runs", "mean": "valid_rate"})
    )
    print(validity.to_string())

    print("\nRuns mit Fehlern / Hinweisen:")
    err_df = df[df["error"].notna() & (df["error"] != "")].copy()
    if len(err_df) == 0:
        print("Keine Fehler")
    else:
        for _, row in err_df.iterrows():
            print(f"  {row['file']} / {row['run_key']}: {str(row['error'])[:140]}")
else:
    print("Keine Ergebnisse gefunden.")

In [ ]:
### Ergebnis-Tabellen und Diagnose-Views
print(f"Gesamt-Runs: {len(df)}")
print(f"Dateien: {df['file'].nunique() if len(df) else 0}")
print(f"Referenzmodus: {'Pseudo-reference (B)' if USE_PSEUDO_REF else 'Manual reference'}")

print("\n1) Alle Runs")
display(df.sort_values(["file", "pipeline", "condition"]).reset_index(drop=True))

print("\n2) Validity nach Pipeline x Condition")
validity_table = (
    df.groupby(["pipeline", "condition"], dropna=False)["valid"]
    .mean()
    .reset_index(name="valid_rate")
    .sort_values(["pipeline", "condition"])
)
display(validity_table)

print("\n3) Invalid Runs")
invalid_runs = df[df["valid"] == False].copy()
display(
    invalid_runs[
        [
            "file", "run_key", "pipeline", "condition",
            "output_exists", "nonempty", "valid",
            "n_ref_eval", "n_hyp_eval",
            "insertions", "deletions", "substitutions",
            "token_error_rate", "valid_vs_transcript", "error",
        ]
    ].sort_values(["file", "pipeline", "condition"]).reset_index(drop=True)
)

valid_df = df[df["wser"].notna()].copy()

if len(valid_df) == 0:
    print("\n Keine validen WSER-Ergebnisse vorhanden.")
else:
    def median_iqr(x: pd.Series) -> str:
        med = x.median()
        iqr = x.quantile(0.75) - x.quantile(0.25)
        return f"{med:.2f} ({iqr:.2f})"

    print("\n4) Table 2: Median WSER (IQR) nach Pipeline × Sprecherzahl × Condition")
    table_wser = (
        valid_df.groupby(["pipeline", "n_speakers_from_filename", "condition"])["wser"]
        .agg(median_iqr)
        .unstack(["n_speakers_from_filename", "condition"])
    )
    table_wser = table_wser.reindex([p for p in PIPELINE_ORDER if p in table_wser.index])
    display(table_wser)

    print("\n4b) Table 3: LLM-Input-Ablation: raw vs segmented")
    llm_df = valid_df[valid_df["pipeline"].isin(["A1_raw", "A1_segmented", "A2_raw", "A2_segmented"])].copy()
    if len(llm_df) == 0:
        print("Keine validen LLM-Ergebnisse vorhanden.")
    else:
        llm_table = (
            llm_df.groupby(["pipeline", "n_speakers_from_filename", "condition"])["wser"]
            .agg(median_iqr)
            .unstack(["n_speakers_from_filename", "condition"])
        )
        llm_table = llm_table.reindex([p for p in PIPELINE_ORDER if p in llm_table.index])
        display(llm_table)

    print("\n5) Table 4: WhisperX-Ablation (C vs D)")
    cd_df = valid_df[valid_df["pipeline"].isin(["C", "D"]) & valid_df["boundary_wser"].notna()].copy()
    if len(cd_df) == 0:
        print("Keine validen C/D-Ergebnisse vorhanden.")
    else:
        table_wx = (
            cd_df.groupby(["pipeline", "condition"])[["wser", "boundary_wser"]]
            .median()
            .round(3)
        )
        display(table_wx)

        for cond in CONDITION_ORDER:
            bwser_c = cd_df[(cd_df["pipeline"] == "C") & (cd_df["condition"] == cond)]["boundary_wser"].median()
            bwser_d = cd_df[(cd_df["pipeline"] == "D") & (cd_df["condition"] == cond)]["boundary_wser"].median()
            if pd.notna(bwser_c) and bwser_c > 0 and pd.notna(bwser_d):
                delta = (bwser_d - bwser_c) / bwser_c * 100
                print(f"Δ Boundary-WSER {cond}: {delta:+.1f}% relativ")

    print("\n6) Table 5: Median RTF")
    rtf_df = df[df["rtf_total"].notna()].copy()
    table_rtf = (
        rtf_df.groupby(["pipeline", "condition"])[["rtf_total", "rtf_post_asr"]]
        .median()
        .round(3)
    )
    desired_idx = [(p, c) for p in PIPELINE_ORDER for c in CONDITION_ORDER if (p, c) in table_rtf.index]
    table_rtf = table_rtf.reindex(desired_idx)
    display(table_rtf)

    print("\n7) LLM Output Validity Rate")
    llm_df = df[df["pipeline"].isin(["A1", "A2"])].copy()
    if len(llm_df):
        llm_validity = llm_df.groupby("pipeline")["valid"].agg(["sum", "count"])
        llm_validity["rate"] = (llm_validity["sum"] / llm_validity["count"]).round(3)
        display(llm_validity)

In [ ]:
### WSER-Stripplot (Paper Figure 2)
plot_df = df[df["wser"].notna()].copy()

if len(plot_df) == 0:
    print("⚠ Keine validen WSER-Werte für den Plot vorhanden.")
else:
    active_pipelines = [p for p in PIPELINE_ORDER if p in plot_df["pipeline"].unique()]
    plot_df["pipeline"] = pd.Categorical(plot_df["pipeline"], categories=active_pipelines, ordered=True)

    fig, axes = plt.subplots(1, 3, figsize=(10, 4), sharey=True)

    for i, n_spk in enumerate([2, 3, 4]):
        ax = axes[i]
        sub = plot_df[plot_df["n_speakers_from_filename"] == n_spk].copy()

        ax.set_title(f"{n_spk} Speakers", fontweight="bold", fontsize=11)

        if len(sub) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_xlabel("")
            ax.set_ylabel("WSER" if i == 0 else "")
            continue

        sns.stripplot(
            data=sub,
            x="pipeline",
            y="wser",
            hue="condition",
            order=active_pipelines,
            hue_order=CONDITION_ORDER,
            palette=CONDITION_COLORS,
            dodge=True,
            jitter=0.15,
            size=6,
            alpha=0.75,
            ax=ax,
            legend=(i == 2),
        )

        medians = sub.groupby(["pipeline", "condition"])["wser"].median()
        pipeline_positions = {p: idx for idx, p in enumerate(active_pipelines)}

        for (pip, cond), med in medians.items():
            if pip not in pipeline_positions:
                continue
            x_base = pipeline_positions[pip]
            x_offset = -0.2 if cond == "unknown" else 0.2
            x_pos = x_base + x_offset
            ax.plot(
                [x_pos - 0.08, x_pos + 0.08],
                [med, med],
                color=CONDITION_COLORS[cond],
                linewidth=2.0,
                solid_capstyle="round",
            )

        ax.set_xlabel("")
        ax.set_ylabel("WSER" if i == 0 else "")
        ax.set_xticklabels(
            [PIPELINE_LABELS.get(p, p) for p in active_pipelines],
            fontsize=8,
            rotation=25,
            ha="right",
        )
        ax.set_ylim(bottom=-0.02)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

    if axes[-1].get_legend():
        axes[-1].get_legend().set_title("")
        axes[-1].get_legend().set_bbox_to_anchor((1.02, 1.0))

    plt.suptitle("Word Speaker Error Rate by Pipeline and Speaker Count", fontsize=12, y=1.03)
    plt.tight_layout()

    fig_path = FIGURES_DIR / "fig2_wser_stripplot.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    print(f"Gespeichert: {fig_path}")
    plt.show()

In [ ]:
### RTF Stacked Bar Chart (Paper Figure 3)
rtf_df = df[(df["condition"] == "unknown") & df["rtf_total"].notna()].copy()

if len(rtf_df) == 0:
    print("Keine RTF-Daten vorhanden.")
else:
    active_pipelines = [p for p in PIPELINE_ORDER if p in rtf_df["pipeline"].unique()]

    components = ["rtf_normalization", "rtf_asr", "rtf_alignment", "rtf_diarization"]
    comp_labels = ["Normalization", "ASR (faster-whisper)", "WhisperX Alignment", "Diarization / LLM"]
    comp_colors = ["#BDBDBD", "#4CAF50", "#FF9800", "#2196F3"]

    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(active_pipelines))
    width = 0.55
    bottom = np.zeros(len(active_pipelines))

    for comp, label, color in zip(components, comp_labels, comp_colors):
        values = (
            rtf_df.groupby("pipeline")[comp]
            .median()
            .reindex(active_pipelines)
            .fillna(0.0)
            .to_numpy()
        )
        ax.bar(
            x,
            values,
            width,
            bottom=bottom,
            label=label,
            color=color,
            edgecolor="white",
            linewidth=0.5,
        )
        bottom += values

    totals = (
        rtf_df.groupby("pipeline")["rtf_total"]
        .median()
        .reindex(active_pipelines)
        .fillna(0.0)
    )

    for i, p in enumerate(active_pipelines):
        total = totals.loc[p]
        ax.text(i, total + 0.01, f"{total:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.axhline(1.0, color="red", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.text(len(active_pipelines) - 0.35, 1.01, "real-time", color="red", fontsize=8, va="bottom")

    ax.set_xticks(x)
    ax.set_xticklabels([PIPELINE_LABELS.get(p, p) for p in active_pipelines], fontsize=9)
    ax.set_ylabel("Real-Time Factor (RTF)")
    ax.set_title("End-to-End RTF Decomposition (unknown speakers)", fontsize=11, fontweight="bold")
    ax.set_ylim(bottom=0)
    ax.legend(loc="upper left", fontsize=8, framealpha=0.95)

    plt.tight_layout()

    fig_path = FIGURES_DIR / "fig3_rtf_stacked.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    print(f"Gespeichert: {fig_path}")
    plt.show()

In [ ]:
### Zusammenfassung
print("=" * 70)
print("ZUSAMMENFASSUNG")
print("=" * 70)

valid_df = df[df["wser"].notna()].copy()

if len(valid_df) == 0:
    print("Keine validen Ergebnisse.")
else:
    print(f"\nDateien: {valid_df['file'].nunique()} | Runs mit WSER: {len(valid_df)}")
    print(f"Referenz: {'Pseudo (Pyannote-Segment)' if USE_PSEUDO_REF else 'Manuell (diarization_ref.txt)'}")

    print("\nBeste Pipeline pro Sprecherzahl (known, Median WSER)")
    for n_spk in sorted(valid_df["n_speakers_from_filename"].dropna().unique()):
        sub = valid_df[
            (valid_df["n_speakers_from_filename"] == n_spk) &
            (valid_df["condition"] == "known")
        ]
        if len(sub) == 0:
            continue
        best = sub.groupby("pipeline")["wser"].median().sort_values()
        best_name = best.index[0]
        best_val = best.iloc[0]
        print(f"  {int(n_spk)} Sprecher: {PIPELINE_LABELS.get(best_name, best_name)} -> WSER {best_val:.3f}")

    print("\nLLM vs. bestes Pyannote (known, Median WSER)")
    for n_spk in sorted(valid_df["n_speakers_from_filename"].dropna().unique()):
        sub = valid_df[
            (valid_df["n_speakers_from_filename"] == n_spk) &
            (valid_df["condition"] == "known")
        ]
        if len(sub) == 0:
            continue
        
        llm_best = sub[
            sub["pipeline"].isin(["A1_raw", "A1_segmented", "A2_raw", "A2_segmented"])
        ].groupby("pipeline")["wser"].median().min()

        pya_best = sub[
            sub["pipeline"].isin(["B", "C", "D"])
        ].groupby("pipeline")["wser"].median().min()
        
        if pd.notna(llm_best) and pd.notna(pya_best):
            delta = llm_best - pya_best
            print(f"  {int(n_spk)} Sprecher: LLM {llm_best:.3f} vs Pyannote {pya_best:.3f} (Δ = {delta:+.3f})")

    print("\nValidity pro Pipeline (gegen Referenz)")
    validity = (
        df.groupby(["pipeline", "condition"])["valid"]
        .mean()
        .unstack("condition")
        .reindex(PIPELINE_ORDER)
        .round(3)
    )
    display(validity)

    print("\nRTF-Übersicht (unknown, Median)")
    rtf_summary = df[df["rtf_total"].notna() & (df["condition"] == "unknown")]
    for p in PIPELINE_ORDER:
        vals = rtf_summary[rtf_summary["pipeline"] == p]["rtf_total"]
        if len(vals) > 0:
            print(f"  {PIPELINE_LABELS.get(p, p)}: RTF {vals.median():.2f}")

    print("\nErzeugte Abbildungen")
    for fig_file in sorted(FIGURES_DIR.glob("*.png")):
        print(f"  {fig_file.name} ({fig_file.stat().st_size / 1024:.0f} KB)")

In [ ]:
### Optional: Debug-Zelle für Token-Mismatches (besonders nützlich für C/D)
DEBUG_PIPELINES = ["C", "D"]   # z.B. ["C"] oder ["C", "D"]
DEBUG_ONLY_INVALID = True
DEBUG_MAX_RUNS = 20

dbg = df[df["pipeline"].isin(DEBUG_PIPELINES)].copy()
if DEBUG_ONLY_INVALID:
    dbg = dbg[dbg["valid"] == False].copy()

dbg = dbg.sort_values(["file", "pipeline", "condition"]).head(DEBUG_MAX_RUNS)

print(f"Debug-Runs: {len(dbg)}")
display(
    dbg[
        [
            "file", "run_key", "pipeline", "condition", "valid",
            "token_error_rate", "n_ref_eval", "n_hyp_eval",
            "insertions", "deletions", "substitutions",
            "valid_vs_transcript",
        ]
    ].reset_index(drop=True)
)

for _, row in dbg.iterrows():
    result_dir = RESULTS_DIR / row["file"]
    ref_path = (
        result_dir / "Pyannote_diarized_segment_known.txt"
        if USE_PSEUDO_REF
        else result_dir / "diarization_ref.txt"
    )
    hyp_path = result_dir / f"{row['run_key']}.txt"

    if not ref_path.exists() or not hyp_path.exists():
        continue

    ref_text = ref_path.read_text(encoding="utf-8")
    hyp_text = hyp_path.read_text(encoding="utf-8")

    mismatch = first_token_mismatch(ref_text, hyp_text, context=8)

    print("\n" + "=" * 90)
    print(f"{row['file']} | {row['run_key']}")
    print(f"Mismatch-Typ: {mismatch.get('tag')}")
    if mismatch.get("tag") != "equal":
        print(f"ref_span: {mismatch['ref_span']} -> {mismatch['ref_tokens']}")
        print(f"hyp_span: {mismatch['hyp_span']} -> {mismatch['hyp_tokens']}")
        print(f"ref_context: {mismatch['ref_context']}")
        print(f"hyp_context: {mismatch['hyp_context']}")
    else:
        print("Kein Mismatch gefunden.")